# Digital Evidence Integrity Verifier

Author: Nnenne Mercy Nwanoruo

## Objective
Develop and evaluate a Python tool for verifying the integrity of
digital files during storage, transfer, and backup restoration,
with potential applications across U.S. criminal justice agencies.

## Implemented capabilities
- Record SHA-256 fingerprints for files and subfolders
- Identify unchanged, modified, missing, and newly added files
- Verify transferred and restored copies
- Report unreadable files and incomplete scans
- Authenticate the reference manifest
- Export dated verification reports

## Evaluation
Use fictional evidence files and controlled changes to test detection,
false alarms, failure handling, and processing performance.

## Scope
The tool compares files against a recorded reference. It does not
establish their original authenticity or identify who changed them.

This is an independent project. Agency endorsement, adoption, and
operational impact will be reported only if supported by evidence.
## Running this demonstration
Run cells from top to bottom. Each full run creates a new demo folder and key.
The signing cell requests a password; do not share it or the private-key file.
Rerunning the manifest/signing cells retains and verifies existing references.
Do not rerun the first setup cell alone and then skip intervening cells.

## Trust and scope
This is an independent demonstration for potential use across U.S. justice settings,
not a deployed agency service, certified forensic product, or complete chain of custody.
The key fingerprint is pinned during trusted initial provisioning. Protect that pin,
the verifier code, and the private signing key outside the evidence storage boundary.
Ed25519 here is not a claim of FIPS validation or CJIS compliance.
Scanning assumes a quiescent folder; metadata checks are not an atomic snapshot or
protection against an attacker concurrently changing filesystem links or contents.
Comparison covers relative paths, file sizes, and bytes, not permissions, alternate
streams, extended attributes, or all filesystem metadata. Reports are unsigned.

## Recorded results
Original Colab results are in `recorded_results/` in the GitHub package. Those
benchmark timings are user-recorded results, not measurements of agency systems.
The original signing key, password, signed manifest, and signature were not supplied
for packaging. This revised notebook creates a fresh demonstration reference.


## Create isolated fictional files


In [ ]:
from pathlib import Path

# Keep evidence files separate from manifests and reports
import tempfile

workspace = Path("/content") if Path("/content").is_dir() else Path.cwd()
demo_parent = workspace / "digital_evidence_demo_runs"
demo_parent.mkdir(parents=True, exist_ok=True)
project_folder = Path(tempfile.mkdtemp(prefix="demo_", dir=demo_parent))
evidence_folder = project_folder / "sample_evidence"

sample_files = {
    "documents/incident_notes.txt": (
        b"FICTIONAL TEST DATA - NOT REAL EVIDENCE\n"
        b"Test case: DEMO-001\n"
        b"Observation: Sample document for integrity verification.\n"
    ),
    "logs/access_log.csv": (
        b"event_id,event_time,event_type\n"
        b"TEST-001,2026-01-01T10:00:00Z,created\n"
        b"TEST-002,2026-01-01T10:05:00Z,reviewed\n"
    ),
    "media/sample_binary.bin": bytes(range(256)) * 4096,
}

# Create files without overwriting existing files
for relative_name, content in sample_files.items():
    file_path = evidence_folder / relative_name
    file_path.parent.mkdir(parents=True, exist_ok=True)

    if not file_path.exists():
        file_path.write_bytes(content)

print("Fictional evidence folder:", evidence_folder)
print("\nFiles currently present:")

for file_path in sorted(evidence_folder.rglob("*")):
    if file_path.is_file():
        relative_name = file_path.relative_to(evidence_folder)
        print(f"{relative_name} — {file_path.stat().st_size:,} bytes")

## Calculate SHA-256 fingerprints


In [ ]:
import hashlib

def calculate_sha256(file_path, chunk_size=1024 * 1024):
    """Calculate a file's SHA-256 hash without changing its contents."""
    if type(chunk_size) is not int or chunk_size <= 0:
        raise ValueError("chunk_size must be a positive integer")

    fingerprint = hashlib.sha256()

    with Path(file_path).open("rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            fingerprint.update(chunk)

    return fingerprint.hexdigest()


# Check the implementation against a known SHA-256 test value
known_hash = (
    "ba7816bf8f01cfea414140de5dae2223"
    "b00361a396177a9cb410ff61f20015ad"
)

assert hashlib.sha256(b"abc").hexdigest() == known_hash

# Check our file-reading function against a direct calculation
sample_path = evidence_folder / "documents/incident_notes.txt"

file_hash = calculate_sha256(sample_path)
direct_hash = hashlib.sha256(sample_path.read_bytes()).hexdigest()

assert file_hash == direct_hash

# Changing chunk size must not change the result
assert calculate_sha256(sample_path, chunk_size=7) == file_hash

print("PASS: Known SHA-256 test value matched.")
print("PASS: File-reading function matched the direct calculation.")
print("PASS: Different chunk sizes produced the same fingerprint.")
print("\nSample document fingerprint:")
print(file_hash)

## Scan folders and report failures


In [ ]:
import os
from datetime import datetime, timezone

def scan_folder(folder):
    """Scan regular files; report failures instead of silently skipping them."""
    root = Path(folder)

    if root.is_symlink() or not root.is_dir():
        raise ValueError("Choose an existing folder, not a symbolic link")

    result = {
        "algorithm": "SHA-256",
        "started_utc": datetime.now(timezone.utc).isoformat(),
        "files": {},
        "issues": [],
    }

    def report_issue(path, reason):
        result["issues"].append({
            "path": str(Path(path).relative_to(root).as_posix()),
            "reason": reason,
        })

    def directory_error(error):
        report_issue(
            error.filename or root,
            f"Directory could not be read: {type(error).__name__}",
        )

    for current, directories, filenames in os.walk(
        root, followlinks=False, onerror=directory_error
    ):
        directories.sort()

        # Do not follow directory links outside the evidence folder
        for name in directories[:]:
            path = Path(current) / name
            if path.is_symlink():
                report_issue(path, "Symbolic-link directory excluded")
                directories.remove(name)

        for name in sorted(filenames):
            path = Path(current) / name
            relative_name = path.relative_to(root).as_posix()

            try:
                if path.is_symlink() or not path.is_file():
                    report_issue(path, "Link or nonregular file excluded")
                    continue

                before = path.stat()
                digest = calculate_sha256(path)
                after = path.stat()

                # Check for observable changes during hashing
                def signature(info):
                    return (
                        info.st_dev, info.st_ino, info.st_size,
                        info.st_mtime_ns, info.st_ctime_ns,
                    )

                if signature(before) != signature(after):
                    report_issue(path, "File changed during hashing; retry needed")
                    continue

                result["files"][relative_name] = {
                    "sha256": digest,
                    "size_bytes": after.st_size,
                }

            except OSError as error:
                report_issue(path, f"File could not be read: {type(error).__name__}")

    result["finished_utc"] = datetime.now(timezone.utc).isoformat()
    result["scan_complete"] = not result["issues"]
    return result


baseline_scan = scan_folder(evidence_folder)

print("Files fingerprinted:", len(baseline_scan["files"]))
print("Scan issues:", len(baseline_scan["issues"]))
print("Scan complete:", baseline_scan["scan_complete"])

for name, details in baseline_scan["files"].items():
    print(f"\n{name}\nSHA-256: {details['sha256']}")

if baseline_scan["issues"]:
    print("\nIssues:", baseline_scan["issues"])

In [ ]:
import zipfile


## Create or retain the reference manifest


In [ ]:
import json

# Only accept a reference scan without reported issues
if not baseline_scan["scan_complete"] or baseline_scan["issues"]:
    raise ValueError("Resolve scan issues before creating the reference")

if not baseline_scan["files"]:
    raise ValueError("Cannot create an empty reference manifest")

manifest = {
    "schema_version": 1,
    "purpose": "Fictional digital-evidence integrity demonstration",
    "algorithm": baseline_scan["algorithm"],
    "scan_started_utc": baseline_scan["started_utc"],
    "scan_finished_utc": baseline_scan["finished_utc"],
    "files": baseline_scan["files"],
}

# Store the reference outside the folder being checked
reference_folder = project_folder / "reference"
reference_folder.mkdir(parents=True, exist_ok=True)

manifest_path = reference_folder / "manifest.json"

manifest_bytes = (
    json.dumps(
        manifest,
        sort_keys=True,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    ) + "\n"
).encode("utf-8")

# Reuse an existing reference byte-for-byte; never silently rebaseline.
if manifest_path.exists():
    manifest_bytes = manifest_path.read_bytes()
    manifest = json.loads(manifest_bytes)
    print("Existing manifest retained without changes.")
else:
    with manifest_path.open("xb") as file:
        file.write(manifest_bytes)

print("Reference manifest saved:", manifest_path)
print("Files recorded:", len(manifest["files"]))
print("Manifest size:", len(manifest_bytes), "bytes")

## Sign or verify the existing reference


In [ ]:
from getpass import getpass

from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric.ed25519 import (
    Ed25519PrivateKey,
)

private_folder = project_folder / "private_keys"
trusted_folder = project_folder / "trusted_keys"

private_folder.mkdir(parents=True, exist_ok=True, mode=0o700)
trusted_folder.mkdir(parents=True, exist_ok=True)

private_key_path = private_folder / "signing_key_encrypted.pem"
public_key_path = trusted_folder / "verification_key.pub"
signature_path = reference_folder / "manifest.sig"

# On a rerun, reuse only a complete reference with the previously pinned key.
existing = [p.exists() for p in (private_key_path, public_key_path, signature_path)]
if any(existing) and not all(existing):
    raise ValueError("Partial key/signature setup. Start a fresh demo from the first cell.")

if all(existing):
    import hmac
    from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PublicKey
    if "EXPECTED_KEY_FINGERPRINT" not in globals():
        raise ValueError("Trusted pin missing. Run from the first cell for a fresh demo.")
    public_bytes = public_key_path.read_bytes()
    trusted_key_fingerprint = hashlib.sha256(public_bytes).hexdigest()
    if not hmac.compare_digest(trusted_key_fingerprint, EXPECTED_KEY_FINGERPRINT):
        raise ValueError("Existing public key does not match the trusted pin")
    Ed25519PublicKey.from_public_bytes(public_bytes).verify(
        signature_path.read_bytes(), manifest_path.read_bytes()
    )
    print("PASS: Existing signed reference verified; no files replaced.")
else:
    password = getpass("Choose a password for the demonstration signing key: ")
    confirmation = getpass("Enter the password again: ")

    if not password or password != confirmation:
        raise ValueError("Passwords must be nonempty and match")

    private_key = Ed25519PrivateKey.generate()
    public_key = private_key.public_key()

    encrypted_private_bytes = private_key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.BestAvailableEncryption(
            password.encode("utf-8")
        ),
    )

    public_bytes = public_key.public_bytes(
        encoding=serialization.Encoding.Raw,
        format=serialization.PublicFormat.Raw,
    )

    # Sign the exact bytes saved on disk
    reference_bytes = manifest_path.read_bytes()
    signature = private_key.sign(reference_bytes)

    with private_key_path.open("xb") as file:
        file.write(encrypted_private_bytes)
    private_key_path.chmod(0o600)

    with public_key_path.open("xb") as file:
        file.write(public_bytes)

    with signature_path.open("xb") as file:
        file.write(signature)

    # Verify the original manifest
    public_key.verify(signature, reference_bytes)
    print("PASS: Original manifest signature verified.")

    # Change a copy in memory; leave the saved manifest untouched
    try:
        public_key.verify(signature, reference_bytes + b" ")
    except InvalidSignature:
        print("PASS: Modified manifest was rejected.")
    else:
        raise AssertionError("Modified manifest was incorrectly accepted")

    trusted_key_fingerprint = hashlib.sha256(public_bytes).hexdigest()

    print("\nPublic-key fingerprint:", trusted_key_fingerprint)
    print("Signature saved:", signature_path)
    print("Encrypted private key saved separately.")

    del password, confirmation, private_key, encrypted_private_bytes
    EXPECTED_KEY_FINGERPRINT = trusted_key_fingerprint


## Authenticate and validate the manifest


In [ ]:
import hmac
import re
from pathlib import PurePosixPath

from cryptography.hazmat.primitives.asymmetric.ed25519 import (
    Ed25519PublicKey,
)

# The signing cell pins the new demo key during initial provisioning.
def _unique_pairs(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError("Duplicate JSON key in manifest")
        result[key] = value
    return result

def _reject_constant(value):
    raise ValueError("Nonstandard JSON numeric constant")

def load_verified_manifest(manifest_file, signature_file,
                           public_key_file, expected_fingerprint):
    """Authenticate and validate a manifest before returning its contents."""
    key_bytes = Path(public_key_file).read_bytes()
    actual_fingerprint = hashlib.sha256(key_bytes).hexdigest()

    if not hmac.compare_digest(
        actual_fingerprint, expected_fingerprint
    ):
        raise ValueError("Verification key does not match the trusted fingerprint")

    key = Ed25519PublicKey.from_public_bytes(key_bytes)
    signed_bytes = Path(manifest_file).read_bytes()
    signature_bytes = Path(signature_file).read_bytes()

    # Raises InvalidSignature if verification fails
    key.verify(signature_bytes, signed_bytes)

    reference = json.loads(signed_bytes, object_pairs_hook=_unique_pairs, parse_constant=_reject_constant)

    if not isinstance(reference, dict):
        raise ValueError("Manifest must be a JSON object")

    if (
        type(reference.get("schema_version")) is not int
        or reference["schema_version"] != 1
        or reference.get("algorithm") != "SHA-256"
    ):
        raise ValueError("Unsupported manifest version or algorithm")

    entries = reference.get("files")

    if not isinstance(entries, dict) or not entries:
        raise ValueError("Manifest must contain file entries")

    for name, details in entries.items():
        # Only accept portable relative paths
        parts = name.split("/")
        if (
            not name
            or any(ord(c) < 32 for c in name)
            or PurePosixPath(name).is_absolute()
            or "\\" in name
            or ":" in name
            or any(part in ("", ".", "..") for part in parts)
        ):
            raise ValueError("Manifest contains an unsafe file path")

        if not isinstance(details, dict):
            raise ValueError("Invalid file entry")

        digest = details.get("sha256")
        size = details.get("size_bytes")

        if (
            not isinstance(digest, str)
            or re.fullmatch(r"[0-9a-f]{64}", digest) is None
            or type(size) is not int
            or size < 0
        ):
            raise ValueError("Invalid fingerprint or file size")

    return reference


verified_manifest = load_verified_manifest(
    manifest_path,
    signature_path,
    public_key_path,
    EXPECTED_KEY_FINGERPRINT,
)

print("PASS: Verification key matched the pinned fingerprint.")
print("PASS: Manifest signature verified.")
print("PASS: Manifest structure and file entries passed validation.")
print("Reference files:", len(verified_manifest["files"]))

## Verify unchanged files


In [ ]:
def verify_folder(folder, manifest_file, signature_file,
                  public_key_file, expected_fingerprint):
    """Authenticate the reference, scan the folder, and compare files."""
    reference = load_verified_manifest(
        manifest_file,
        signature_file,
        public_key_file,
        expected_fingerprint,
    )

    current = scan_folder(folder)
    expected_files = reference["files"]
    observed_files = current["files"]
    comparisons = []

    for name in sorted(set(expected_files) | set(observed_files)):
        expected = expected_files.get(name)
        observed = observed_files.get(name)

        if expected is None:
            status = "NEW"

        elif observed is None:
            # An incomplete scan cannot reliably establish absence
            status = (
                "MISSING" if current["scan_complete"]
                else "NOT_VERIFIED"
            )

        elif (
            expected["sha256"] == observed["sha256"]
            and expected["size_bytes"] == observed["size_bytes"]
        ):
            status = "UNCHANGED"

        else:
            status = "MODIFIED"

        comparisons.append({
            "path": name,
            "status": status,
            "expected_sha256": expected["sha256"] if expected else None,
            "observed_sha256": observed["sha256"] if observed else None,
            "expected_size": expected["size_bytes"] if expected else None,
            "observed_size": observed["size_bytes"] if observed else None,
        })

    if not current["scan_complete"]:
        outcome = "INCOMPLETE"
    elif any(row["status"] != "UNCHANGED" for row in comparisons):
        outcome = "DIFFERENCES_DETECTED"
    else:
        outcome = "VERIFIED_MATCH"

    return {
        "reference_authenticated": True,
        "verification_key_fingerprint": expected_fingerprint,
        "scan_started_utc": current["started_utc"],
        "scan_finished_utc": current["finished_utc"],
        "outcome": outcome,
        "files": comparisons,
        "scan_issues": current["issues"],
    }


first_report = verify_folder(
    evidence_folder,
    manifest_path,
    signature_path,
    public_key_path,
    EXPECTED_KEY_FINGERPRINT,
)

print("Outcome:", first_report["outcome"])

for row in first_report["files"]:
    print(f"{row['status']}: {row['path']}")

print("Scan issues:", len(first_report["scan_issues"]))

## Detect controlled file changes


In [ ]:
import shutil
import tempfile

# Create a separate test copy
test_run_folder = Path(tempfile.mkdtemp(
    prefix="change_test_",
    dir=project_folder,
))

changed_folder = test_run_folder / "evidence_copy"
shutil.copytree(evidence_folder, changed_folder)

# 1. Change exactly one byte without changing the file size
modified_path = changed_folder / "documents/incident_notes.txt"
content = bytearray(modified_path.read_bytes())
content[-1] ^= 1
modified_path.write_bytes(content)

# 2. Remove a file from the test copy
(changed_folder / "logs/access_log.csv").unlink()

# 3. Add a new fictional file
(changed_folder / "documents/extra_note.txt").write_text(
    "FICTIONAL TEST FILE: added after the reference scan.\n",
    encoding="utf-8",
)

change_report = verify_folder(
    changed_folder,
    manifest_path,
    signature_path,
    public_key_path,
    EXPECTED_KEY_FINGERPRINT,
)

expected_statuses = {
    "documents/incident_notes.txt": "MODIFIED",
    "logs/access_log.csv": "MISSING",
    "documents/extra_note.txt": "NEW",
    "media/sample_binary.bin": "UNCHANGED",
}

actual_statuses = {
    row["path"]: row["status"]
    for row in change_report["files"]
}

assert actual_statuses == expected_statuses, (
    f"Unexpected results: {actual_statuses}"
)
assert change_report["outcome"] == "DIFFERENCES_DETECTED"
assert not change_report["scan_issues"]

print("Outcome:", change_report["outcome"])

for row in change_report["files"]:
    print(f"{row['status']}: {row['path']}")

print("\nPASS: All four file statuses matched expectations.")
print("PASS: A one-byte change was detected despite unchanged file size.")

## Reject altered references and untrusted keys


In [ ]:
security_test_folder = Path(tempfile.mkdtemp(
    prefix="reference_security_test_",
    dir=project_folder,
))

# Test copies containing deliberate changes
altered_manifest = security_test_folder / "altered_manifest.json"
altered_manifest.write_bytes(manifest_path.read_bytes() + b" ")

altered_signature = security_test_folder / "altered_manifest.sig"
signature_copy = bytearray(signature_path.read_bytes())
signature_copy[0] ^= 1
altered_signature.write_bytes(signature_copy)

# Generate an unrelated demonstration key
unrelated_public_key = Ed25519PrivateKey.generate().public_key()
unrelated_key_path = security_test_folder / "unrelated_key.pub"
unrelated_key_path.write_bytes(
    unrelated_public_key.public_bytes(
        encoding=serialization.Encoding.Raw,
        format=serialization.PublicFormat.Raw,
    )
)


def expect_rejection(label, manifest_file, signature_file,
                     key_file, expected_error):
    try:
        verify_folder(
            evidence_folder,
            manifest_file,
            signature_file,
            key_file,
            EXPECTED_KEY_FINGERPRINT,
        )
    except expected_error:
        print(f"PASS: {label} — verification stopped.")
    else:
        raise AssertionError(f"FAIL: {label} was accepted")


expect_rejection(
    "Altered manifest",
    altered_manifest,
    signature_path,
    public_key_path,
    InvalidSignature,
)

expect_rejection(
    "Altered signature",
    manifest_path,
    altered_signature,
    public_key_path,
    InvalidSignature,
)

expect_rejection(
    "Untrusted public key",
    manifest_path,
    signature_path,
    unrelated_key_path,
    ValueError,
)

# Confirm the original reference still works
original_report = verify_folder(
    evidence_folder,
    manifest_path,
    signature_path,
    public_key_path,
    EXPECTED_KEY_FINGERPRINT,
)

assert original_report["outcome"] == "VERIFIED_MATCH"
print("PASS: Original reference still verifies successfully.")

## Simulate a file-read failure


In [ ]:
import sys
from unittest.mock import patch

original_hash_function = calculate_sha256
blocked_file = evidence_folder / "logs/access_log.csv"


def simulate_read_failure(file_path, chunk_size=1024 * 1024):
    if Path(file_path) == blocked_file:
        raise PermissionError("Simulated file-read failure")

    return original_hash_function(file_path, chunk_size)


# Temporarily substitute the hashing function during this test only
with patch.object(
    sys.modules[scan_folder.__module__],
    "calculate_sha256",
    side_effect=simulate_read_failure,
):
    failure_report = verify_folder(
        evidence_folder,
        manifest_path,
        signature_path,
        public_key_path,
        EXPECTED_KEY_FINGERPRINT,
    )

statuses = {
    row["path"]: row["status"]
    for row in failure_report["files"]
}

assert failure_report["outcome"] == "INCOMPLETE"

assert statuses == {
    "documents/incident_notes.txt": "UNCHANGED",
    "logs/access_log.csv": "NOT_VERIFIED",
    "media/sample_binary.bin": "UNCHANGED",
}

assert failure_report["scan_issues"] == [{
    "path": "logs/access_log.csv",
    "reason": "File could not be read: PermissionError",
}]

assert calculate_sha256 is original_hash_function

print("Outcome:", failure_report["outcome"])

for row in failure_report["files"]:
    print(f"{row['status']}: {row['path']}")

print("\nRecorded scan issues:")
for issue in failure_report["scan_issues"]:
    print(f"{issue['path']}: {issue['reason']}")

print("\nPASS: Read failure prevented a successful overall result.")
print("PASS: Unreadable file was NOT_VERIFIED, not MISSING.")
print("PASS: Original hashing function restored.")

## Verify a local destination copy


In [ ]:
# Simulate a destination folder after a transfer or backup restoration
destination_run = Path(tempfile.mkdtemp(
    prefix="destination_test_",
    dir=project_folder,
))

destination_folder = destination_run / "restored_evidence"
shutil.copytree(evidence_folder, destination_folder)

destination_report = verify_folder(
    destination_folder,
    manifest_path,
    signature_path,
    public_key_path,
    EXPECTED_KEY_FINGERPRINT,
)

assert destination_report["outcome"] == "VERIFIED_MATCH"
assert len(destination_report["files"]) == 3
assert all(
    row["status"] == "UNCHANGED"
    for row in destination_report["files"]
)
assert not destination_report["scan_issues"]

print("Destination:", destination_folder)
print("Outcome:", destination_report["outcome"])

for row in destination_report["files"]:
    print(f"{row['status']}: {row['path']}")

print("\nPASS: Destination files matched the authenticated reference.")
print("PASS: Verification worked from a different parent folder.")

## Measure hashing performance


In [ ]:
import time
import pandas as pd

benchmark_folder = Path(tempfile.mkdtemp(
    prefix="benchmark_",
    dir=project_folder,
))

# Repeatable fictional binary content: one 1 MiB block
block = bytes(range(256)) * 4096
benchmark_results = []

for size_mib in [1, 16, 64]:
    file_path = benchmark_folder / f"sample_{size_mib}MiB.bin"
    expected_hash = hashlib.sha256()

    # Generate the file and its expected hash before timing
    with file_path.open("xb") as file:
        for _ in range(size_mib):
            file.write(block)
            expected_hash.update(block)

    elapsed_times = []

    # Three timed runs per file
    for _ in range(3):
        started = time.perf_counter()
        observed_hash = calculate_sha256(file_path)
        elapsed = time.perf_counter() - started

        assert observed_hash == expected_hash.hexdigest()
        elapsed_times.append(elapsed)

    median_seconds = sorted(elapsed_times)[1]

    benchmark_results.append({
        "file_size_mib": size_mib,
        "runs": 3,
        "median_seconds": median_seconds,
        "throughput_mib_per_second": size_mib / median_seconds,
        "all_hashes_matched": True,
    })

benchmark_table = pd.DataFrame(benchmark_results)

print("Hashing performance — current Colab session")
display(benchmark_table.round(4))
print("PASS: All nine timed runs matched the expected fingerprints.")

## Export reports


In [ ]:
import zipfile
import platform
from importlib.metadata import version

export_folder = Path(tempfile.mkdtemp(
    prefix="verification_reports_",
    dir=project_folder,
))

reports = {
    "unchanged_folder": first_report,
    "changed_folder": change_report,
    "simulated_read_failure": failure_report,
    "destination_copy": destination_report,
}

for name, report in reports.items():
    # Detailed report, including authentication and scan issues
    (export_folder / f"{name}.json").write_text(
        json.dumps(report, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )

    # File comparison table for easy review
    pd.DataFrame(report["files"]).to_csv(
        export_folder / f"{name}.csv",
        index=False,
    )

benchmark_table.to_csv(
    export_folder / "hashing_benchmark.csv",
    index=False,
)

environment_details = {
    "exported_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "platform": platform.system(),
    "pandas_version": pd.__version__,
    "cryptography_version": version("cryptography"),
    "hashing_chunk_size_bytes": 1024 * 1024,
    "benchmark_runs_per_size": 3,
    "benchmark_note": (
        "Local Colab hashing measurements. File creation excluded. "
        "Caching may affect timings. Not full verification throughput."
    ),
}

(export_folder / "environment.json").write_text(
    json.dumps(environment_details, indent=2) + "\n",
    encoding="utf-8",
)

(export_folder / "README.txt").write_text(
    "DIGITAL EVIDENCE INTEGRITY VERIFIER — DEMONSTRATION RESULTS\n\n"
    "All evidence files used in these experiments were fictional.\n"
    "JSON reports contain overall outcomes and scan issues.\n"
    "CSV reports contain file-level comparisons.\n"
    "The read-failure experiment used a simulated PermissionError.\n"
    "The destination-copy experiment used a local folder copy.\n"
    "Benchmark measurements apply only to the recorded environment.\n\n"
    "These exported reports are not digitally signed. "
    "Manifest authentication does not authenticate the reports themselves.\n"
    "UTC timestamps are runtime clock readings, not trusted timestamps.\n"
    "No evidence files or private signing keys are included.\n",
    encoding="utf-8",
)

# Bundle only the explicitly exported report files
bundle_path = project_folder / f"{export_folder.name}.zip"

with zipfile.ZipFile(bundle_path, "x", zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted(export_folder.iterdir()):
        if file_path.is_file():
            archive.write(file_path, arcname=file_path.name)

print("Reports exported:", len(list(export_folder.iterdir())))
print("Download:", bundle_path.name)

try:
    from google.colab import files
except ImportError:
    print("Local notebook: retrieve the ZIP from the printed project folder.")
else:
    files.download(str(bundle_path))